# VIscaNEr Stage 2B — joint DINOv3-B + fusion tuning

Starts from the best Stage-2A Fusion MLP. The last two blocks and retrieval heads of both DINOv3-B branches are trained with the final listwise fusion loss plus separate label and bottle retrieval losses. Candidate unions are re-mined every three epochs. OCR remains frozen/missing-aware.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_fusion')
OUTPUT = Path('/kaggle/working/fusion_stage2b')
OCR_CSV = None  # must match the OCR CSV used by Stage 2A, if any
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

def dataset_root(slug, marker):
    roots = set()
    for match in INPUT.rglob(marker):
        for parent in (match, *match.parents):
            if parent.name == slug:
                roots.add(parent); break
    if len(roots) != 1:
        raise RuntimeError(f'Attach exactly one dataset {slug} containing {marker}; found: {sorted(roots)}')
    return roots.pop()

def unique_file(filename, required_parent_token):
    matches = [p for p in INPUT.rglob(filename) if required_parent_token in p.as_posix()]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {filename} from {required_parent_token}; found: {matches}')
    return matches[0]

code = dataset_root('viscaner-fusion-stage2-code', 'code_sha256.json')
hard = dataset_root('viscaner-fusion-hardset-v1', 'manifest.csv')
labels = dataset_root('viscaner-dinov3-data', 'crops_metadata.csv')
bottles = dataset_root('viscaner-bottle-classifier-data', 'audit_metadata.csv')
label_checkpoint = unique_file('best.pt', 'dinov3_vitb16_labels_hard_finetune')
bottle_checkpoint = unique_file('best.pt', 'dinov3_vitb16_bottles_hard_finetune')
fusion_checkpoint = unique_file('fusion_mlp_best.pt', 'fusion_stage2a')
feature_cache = unique_file('fusion_features.pt', 'fusion_stage2a')
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
hard = materialize_dataset_dirs(hard, ('label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs', 'crops'), cache)
bottles = materialize_dataset_dirs(bottles, ('refs', 'crops'), cache)
for required in (hard/'manifest.csv', hard/'label_crops', hard/'bottle_crops', labels/'refs', labels/'crops', bottles/'refs', bottles/'crops', label_checkpoint, bottle_checkpoint, fusion_checkpoint, feature_cache):
    if not required.exists():
        raise FileNotFoundError(required)
print('SETUP | Stage-I x2 and Stage-2A outputs resolved', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'rapidfuzz>=3,<4', 'Unidecode>=1.3,<2'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), '| joint batch=12', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'fusion_stage2.train_fusion',
    '--stage', 'joint', '--manifest', str(hard/'manifest.csv'), '--hard-root', str(hard),
    '--label-data-root', str(labels), '--bottle-data-root', str(bottles),
    '--label-refs-root', str(labels/'refs'), '--bottle-refs-root', str(bottles/'refs'),
    '--label-checkpoint', str(label_checkpoint), '--bottle-checkpoint', str(bottle_checkpoint),
    '--fusion-checkpoint', str(fusion_checkpoint), '--feature-cache', str(feature_cache),
    '--output-dir', str(OUTPUT), '--device', 'cuda', '--top-k', '10',
    '--unfreeze-last-blocks', '2', '--remine-every', '3',
    '--joint-epochs', '18', '--min-epochs', '6', '--patience', '5',
    '--joint-batch-size', '12', '--eval-batch-size', '48',
    '--fusion-lr', '1e-4', '--head-lr', '1e-5', '--backbone-lr', '5e-7',
    '--label-loss-weight', '0.25', '--bottle-loss-weight', '0.25', '--amp',
]
if OCR_CSV:
    command += ['--ocr-csv', str(OCR_CSV)]
print('TRAINING | Stage 2B: last 2 blocks of both DINO-B + MLP; OCR frozen; monitor=hard_val Recall@1', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'joint_final_metrics.json').read_text())
print('FINAL STAGE 2B |', json.dumps(summary, ensure_ascii=False), flush=True)
